In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
%run ../gs_slimming.py
print("="*120)
print("gs_slimming done.")
print("="*120)
%run flattening.py
print("="*120)
print("flattening done.")
print("="*120)

Mismatches: {'ViacomCBS_ESG Report_2020-2021_vFINAL.pdf'}

No. of gold_standard reports: 139

No. of downloadable reports: 113

No. of reports in gs_slim: 54


status
partial     45
complete     9
Name: count, dtype: int64

scopes_present
[1, 2lb]            18
[1, 2lb, 3]         15
[1, 2lb, 2mb, 3]     9
[1, 2lb, 2mb]        6
[1, 2mb, 3]          1
[1, 3]               1
[1]                  1
[2lb, 3]             1
b]                1
[3]                  1
Name: count, dtype: int64
gs_slimming done.

  Flattening 53 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineA/PipelineA-Answers.csv
  Reports    : 52
  Zeilen     : 643
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineA/PipelineA-Answers.csv

Total Errors: 0
[OK] No errors detected.
flattening done.


## Import slimmed down Gold-Standard

In [3]:
gs = pd.read_json("../gs_slim.json")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import all 3x2=6 Extractions

In [4]:
answers= pd.read_csv("./PipelineA-Answers.csv")


df_to_merge = [
    (answers, "_A"),
]

In [5]:
# Year normalization RegEx-Script

import re

def normalize_year(raw: str, years_present: set[int] | None = None) -> tuple[int | None, str]:
    """Map a raw fiscal-year label to a calendar year. Returns (year, rule_applied)."""
    label = raw.strip().upper().removeprefix("FY").strip()

    if re.fullmatch(r"\d{4}", label):
        return int(label), "plain"
    if re.fullmatch(r"\d{2}", label):
        return 2000 + int(label), "fy_2digit"

    m = re.fullmatch(r"(\d{4})/(\d{1,4})", label)
    if m:
        left, right = m.groups()
        if len(right) == 4:
            return int(left), "range_later_year"  # 2021/2022 -> 2021
        candidates = {int(left), int(left) + 1}
        if years_present:
            hit = candidates & years_present
            if len(hit) == 1:
                return hit.pop(), "fy_end_month_context"
        return int(left), "fy_end_month_fallback"

    return None, "unparseable"

In [6]:
# Get years from extractions 
years_in_extractions = set()
for df, _ in df_to_merge:
    years_in_extractions.update(df["year"].unique().tolist())

# Create ynrom DataFrames to preserve the original ones
answers_ynorm = answers.copy()

df_to_merge_ynorm = [
    (answers_ynorm, "_A")
]

# Now normalization via the RegEx Script from above
for df, _ in df_to_merge_ynorm:
    df["year"], df["year_rule"] = zip(*df["year"].apply(
        normalize_year,
        years_present=years_in_extractions
    ))

In [7]:
#print("Think ok:", all(answers["report_name"].unique() == gs["report_name"].unique()))
#print()

#print(answers["report_name"].isin(gs["report_name"]).all())
#print(gs["report_name"].isin(answers["report_name"]).all())

in_think1_not_gs  = sorted(set(answers["report_name"]) - set(gs["report_name"]))
in_gs_not_answers  = sorted(set(gs["report_name"])     - set(answers["report_name"]))

print(f"In answers, nicht in GS: {len(in_think1_not_gs)} ==> {"OK" if len(in_think1_not_gs) == 0 else "NOK"}")
for r in sorted(in_think1_not_gs): print(" ", r)

print(f"\nIn GS, nicht in answers: {len(in_gs_not_answers)} ==> {"OK" if len(in_gs_not_answers) == 0 else "NOK"}")
for r in sorted(in_gs_not_answers): print(" ", r)

In answers, nicht in GS: 0 ==> OK

In GS, nicht in answers: 2 ==> NOK
  innospec inc_2020_report
  nok corporation_2021_report


## Matching Extractions to Gold-Standard Scheme

### Before ynrom

In [8]:
years = set()
year_report = []

for df, _ in df_to_merge:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

['2012', '2014', '2015', '2016', '2017', '2017_pro_rata', '2018', '2019', '2019/2020', '2020', '2020/2021', '2021', '2021/2022', '2022', 'FY 2021', 'FY 2022', 'FY16', 'FY17', 'FY18', 'FY19', 'FY20', 'FY2017', 'FY2018', 'FY2019', 'FY2020', 'FY2020/3', 'FY2021', 'FY2022']


### After ynrom

In [9]:
years = set()
year_report = []

for df, _ in df_to_merge_ynorm:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements
    
print(sorted(years))

[2012.0, 2014.0, 2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0, 2021.0, 2022.0, nan]


## Mapping special occurences

In [10]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



Total NaN nach Konversion: 0


In [11]:

# Prints out only if sth. goes wrong
for df, sf in df_to_merge_ynorm:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. ursprüngliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")



Total NaN nach Konversion: 0


## Merging Extraction Values and Gold-Standard

In [12]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [13]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   year             2208 non-null   str    
 2   scope            2208 non-null   str    
 3   page             490 non-null    float64
 4   value            489 non-null    float64
 5   unit             488 non-null    str    
 6   unit_normalized  488 non-null    str    
 7   label            489 non-null    str    
 8   status           2208 non-null   str    
 9   scopes_present   2208 non-null   object 
 10  years_present    2208 non-null   object 
 11  value_A          456 non-null    object 
 12  unit_A           456 non-null    object 
 13  label_A          456 non-null    object 
dtypes: float64(2), object(5), str(7)
memory usage: 241.6+ KB


In [14]:
merged_ynorm = gs.copy()
merged_ynorm["year"] = merged_ynorm["year"].astype(int)

for df, sf in df_to_merge_ynorm:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged_ynorm = pd.merge(merged_ynorm, agg, on=merge_on, how="left")

merged_ynorm.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   year             2208 non-null   int64  
 2   scope            2208 non-null   str    
 3   page             490 non-null    float64
 4   value            489 non-null    float64
 5   unit             488 non-null    str    
 6   unit_normalized  488 non-null    str    
 7   label            489 non-null    str    
 8   status           2208 non-null   str    
 9   scopes_present   2208 non-null   object 
 10  years_present    2208 non-null   object 
 11  value_A          530 non-null    object 
 12  unit_A           530 non-null    object 
 13  label_A          530 non-null    object 
dtypes: float64(2), int64(1), object(5), str(6)
memory usage: 241.6+ KB


#### Rows where no value is present must be from Type "NaN" for better analysis; not "None" because the object is missing.

In [15]:
pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

for col in pipeline_cols:
    merged[col] = merged[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    merged_ynorm[col] = merged_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

### Rearranging Columns

In [16]:

gs_cols = [
    'report_name', 'status', 'scopes_present', 'years_present', # Per-Report granularity
    'year', 'scope',                                            # Per category
    'page', 'value', 'unit',                                    # About the value as-is in the report
    'unit_normalized', 'label',                                 # Additional information
]

pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

merged = merged[gs_cols + pipeline_cols]
merged_ynorm = merged_ynorm[gs_cols + pipeline_cols]

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   status           2208 non-null   str    
 2   scopes_present   2208 non-null   object 
 3   years_present    2208 non-null   object 
 4   year             2208 non-null   str    
 5   scope            2208 non-null   str    
 6   page             490 non-null    float64
 7   value            489 non-null    float64
 8   unit             488 non-null    str    
 9   unit_normalized  488 non-null    str    
 10  label            489 non-null    str    
 11  value_A          456 non-null    object 
 12  unit_A           456 non-null    object 
 13  label_A          456 non-null    object 
dtypes: float64(2), object(5), str(7)
memory usage: 241.6+ KB


## Saving created DataFrame

In [17]:
merged.to_json("PipelineA.json", index=False, orient="records", indent=4)
merged_ynorm.to_json("PipelineA_ynorm.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_csv("gs_extractions_raw.csv")   # Lists as Strings. Needs ast.literal_eval
# pd.read_json("gs_extractions_raw.json", orient="records")  # Lists instant usable
